In [ ]:
import os
import json
from dotenv import load_dotenv
from google import genai

# Setup API keys, this uses Gemini
load_dotenv()
api_key = os.getenv("API_KEY")
client = genai.Client(api_key = api_key)

# Load folder containing the JSON files with the inputs. This folder must contain a file called expected.txt with the expected output
evaluation_folder = "" # Replace this with the name of the folder to evaluate 
with open(rf"{evaluation_folder}/expected.txt", "r") as f:
    expected_output = f.read()

# System prompt
system_prompt = f"Your role is to evaluate the response of another LLM model in response to user inputs. You will be given 2 things, "\
"a JSON of the conversation between the user and the LLM labelled conversation and the outputted response as a result of the conversation labelled report.\n"\
"Your role is to do the following, evaluate how good the response is relative to a description of the expected response for that type of user input. "\
"Next, give a score between 0 to 10 of how good the response is, then give a reason for your score in up to a maximum of 2 sentences.\n"\
"You shall give your response in the following manner:\nScore: (Your score)/10\nReason:\n\n"\
"The expected response for the following user inputs are as follows: " + expected_output

config = genai.types.GenerateContentConfig(
    system_instruction = system_prompt,
    temperature = 0.1
)

print(system_prompt)

In [ ]:
for i in os.listdir(evaluation_folder):
    if ".json" in i:
        #print(i)
        with open(rf"{evaluation_folder}/{i}", "r") as f:
            session = json.load(f)

        if "evaluated" not in session.keys():
            conversation = session["messages"]
            report = session["report"]

            # Start new conversation and send these information to the LLM
            chat = client.chats.create(
                model = "gemini-3.1-flash-lite",
                config = config
            )
            response = chat.send_message(f"conversation: {conversation}\nReport: {report}")
            print(response.text)

            # Store this response in a .txt file
            with open(rf"{evaluation_folder}/results.txt", "a") as f:
                f.write(f"{evaluation_folder}/{i}\n{response.text}\n\n")

            # Prevent file from being predicted on again by adding a new key-value pair evaluated
            session.update({"evaluated" : True})
            with open(rf"{evaluation_folder}/{i}", "w") as f:
                json.dump(session, f, indent = 4)

        else:
            print(f"{evaluation_folder}/{i} skipped as already evaluated")